# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zezo-Elkafoury/Flyrank-internship-assignment-1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule prioritizes pages that are both stale and visible in search.

A page is considered a refresh-review candidate when:

- days_since_last_update >= 180, and
impressions_90d >= 500.


The rule gives these pages a higher score because an old page with meaningful search visibility represents a potentially valuable opportunity for content review.


The baseline uses one reason code:

stale_visible_page — the page has been stale for at least 180 days and has at least 500 impressions over the trailing 90-day window.

The action label for this reason code is:

REVIEW_REFRESH


Pages that do not satisfy the rule receive:

score: 0
reason code: no_priority_signal
action: NO_ACTION

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Make a copy so the original data is unchanged
import pandas as pd
import os

df = pd.read_csv('https://raw.githubusercontent.com/zezo-Elkafoury/Flyrank-internship-assignment-1/main/data/raw/content_refresh_anonymized.csv')
baseline = df.copy()

# Start every page with no priority
baseline["score"] = 0

# Apply the baseline rule
stale_visible = (
    (baseline["days_since_last_update"] >= 180)
    & (baseline["impressions_90d"] >= 500)
)

baseline.loc[stale_visible, "score"] = 1

# Reason code
baseline["reason_code"] = "no_priority_signal"
baseline.loc[stale_visible, "reason_code"] = "stale_visible_page"

# Action label
baseline["action"] = "NO_ACTION"
baseline.loc[stale_visible, "action"] = "REVIEW_REFRESH"

# Rank the queue
baseline = baseline.sort_values(
    ["score", "impressions_90d", "content_age_days"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Add rank
baseline.insert(0, "rank", baseline.index + 1)

# Keep the output focused
queue = baseline[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "impressions_90d",
        "days_since_last_update",
        "content_age_days"
    ]
]

# Create the directory if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Write the required CSV
queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(f"Rows written: {len(queue):,}")
print("\nAction counts:")
print(queue["action"].value_counts())

print("\nTop 20:")
display(queue.head(20))


Rows written: 30,000

Action counts:
action
NO_ACTION         29983
REVIEW_REFRESH       17
Name: count, dtype: int64

Top 20:


,rank,content_id,client_id,score,reason_code,action,impressions_90d,days_since_last_update,content_age_days
0,1,content_cf56e2e2e282,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,61678,194,231
1,2,content_7368877ea310,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,59472,194,231
2,3,content_1bfaa38ff26c,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,25715,194,231
3,4,content_0a91db491d14,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,13299,193,231
4,5,content_5feee3994adb,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,7812,194,231
5,6,content_c2d929d83eaa,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,7558,193,231
6,7,content_b16bd7307b39,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,4590,194,231
7,8,content_fe16a55cd13d,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,4556,194,231
8,9,content_ecb6215e79fd,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,4429,194,231
9,10,content_928af3e22c80,client_7f2253d7e2,1,stale_visible_page,REVIEW_REFRESH,1697,193,231


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

**Rank 1:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 61,678 impressions and has not been updated for 194 days, so it strongly matches the baseline rule. My confidence is medium because age and visibility alone do not prove that the content needs updating. The recommendation could be wrong if the page is still accurate, relevant, and satisfying the current search intent.

**Rank 2:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 59,472 impressions and has not been updated for 194 days, making it a strong candidate under the rule. My confidence is medium because the rule does not directly measure content quality. It could be wrong if the page is already performing well and does not need a refresh.

**Rank 3:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 25,715 impressions and is 194 days since its last update. This gives it meaningful search visibility while also meeting the staleness threshold. My confidence is medium because the page may still be relevant despite its age. The recommendation could be wrong if there is no meaningful content gap to address.

**Rank 4:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 13,299 impressions and has not been updated for 193 days. It therefore meets both conditions of the baseline. My confidence is medium because the rule does not consider whether the page is actually underperforming. The recommendation could be wrong if its current content remains effective.

**Rank 5:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 7,812 impressions and has been unchanged for 194 days. This makes it a reasonable candidate for manual review. My confidence is medium because an older page is not necessarily an outdated page. The recommendation could be wrong if the content is evergreen and still fully satisfies user needs.

**Rank 6:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 7,558 impressions and has not been updated for 193 days. It has enough visibility to make a potential refresh worth investigating. My confidence is medium because the rule cannot determine whether the cause of any performance limitation is the content itself. The recommendation could be wrong if other factors, such as search competition, are responsible.

**Rank 7:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 4,590 impressions and is 194 days since its last update. It satisfies both conditions of the baseline. My confidence is medium because the rule does not measure actual content decay. The recommendation could be wrong if the page has remained useful and relevant despite not being updated recently.

**Rank 8:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 4,556 impressions and has not been updated for 194 days. Its visibility and age make it a reasonable page to investigate. My confidence is medium because the baseline only identifies a potential opportunity rather than proving one exists. The recommendation could be wrong if the existing content already satisfies search intent.

**Rank 9:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 4,429 impressions and has been unchanged for 194 days. It therefore meets the baseline's two conditions. My confidence is medium because high visibility does not necessarily indicate that the page requires an update. The recommendation could be wrong if its current performance is stable and the content remains appropriate.

**Rank 10:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 1,697 impressions and has not been updated for 193 days. It qualifies because it has enough visibility and exceeds the staleness threshold. My confidence is medium because the rule does not include a direct measure of content quality. The recommendation could be wrong if the page is still useful and accurate.

**Rank 11:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 1,408 impressions and has not been updated for 183 days. It meets the baseline conditions, although its visibility is much lower than the highest-ranked pages. My confidence is medium because the potential value of refreshing the page is less clear. The recommendation could be wrong if the page has limited strategic value or does not have a meaningful content issue.

**Rank 12:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 1,316 impressions and has not been updated for 194 days. It meets both conditions of the rule and is therefore placed in the review queue. My confidence is medium because the baseline does not establish that the content has actually become outdated. The recommendation could be wrong if the page is evergreen and remains accurate.

**Rank 13:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 954 impressions and has not been updated for 301 days, making it one of the oldest qualifying pages in the queue. My confidence is medium because its age provides a reason to investigate, but not proof that a refresh will help. The recommendation could be wrong if the page is intentionally stable or its information remains current.

**Rank 14:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 828 impressions and has not been updated for 194 days. It qualifies because it has sufficient visibility and exceeds the 180-day staleness threshold. My confidence is medium because the rule does not account for search intent or content quality. The recommendation could be wrong if the page already meets users' needs.

**Rank 15:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 821 impressions and has not been updated for 301 days. Its age makes it worth investigating, while its search visibility means that a successful improvement could potentially matter. My confidence is medium because age alone does not prove that the content is outdated. The recommendation could be wrong if the page remains accurate and useful.

**Rank 16:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 545 impressions and has not been updated for 183 days. It narrowly exceeds the visibility threshold while also meeting the staleness condition. My confidence is medium to low because it has relatively little search visibility compared with the higher-ranked candidates. The recommendation could be wrong if the potential benefit of refreshing this page is too small to justify the effort.

**Rank 17:** The recommended action is `REVIEW_REFRESH` with the reason code `stale_visible_page`. The page has 533 impressions and has not been updated for 183 days, so it just meets the two baseline thresholds. My confidence is medium to low because this is a borderline candidate based on visibility. The recommendation could be wrong if the page has little strategic value or if there is no actual content problem to solve.

**Rank 18:** The recommended action is `NO_ACTION` with the reason code `no_priority_signal`. Although the page has an extremely high 517,715 impressions, it was updated only 104 days ago and therefore does not meet the 180-day staleness condition. My confidence in the rule's decision is high because the page clearly fails the staleness requirement. However, the baseline could still miss an opportunity if the page has another problem that requires attention before it becomes stale.

**Rank 19:** The recommended action is `NO_ACTION` with the reason code `no_priority_signal`. The page has 517,109 impressions, but it was updated only 22 days ago. It therefore does not meet the staleness requirement. My confidence in the baseline decision is high because refreshing such a recently updated page would generally be premature under this rule. However, the rule could miss an opportunity if the page has a serious performance problem unrelated to content age.

**Rank 20:** The recommended action is `NO_ACTION` with the reason code `no_priority_signal`. The page has 509,252 impressions but was updated only 20 days ago, so it does not qualify as stale. My confidence in the rule's decision is high because it clearly falls outside the baseline's refresh criteria. The recommendation could be wrong if there is another reason to review the page that is not captured by content age and search visibility.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak picks

Some of the baseline's recommendations look weaker than others. **Rank 17** is a weak pick because it has only 533 impressions over 90 days and 183 days since its last update. It technically satisfies both thresholds, but the relatively low visibility means that the potential value of a refresh may be limited. **Rank 16** is another weak pick for the same reason: it has only 545 impressions and is 183 days old, so it only narrowly qualifies for the baseline. These pages demonstrate that the fixed thresholds can produce candidates that are technically valid but may not be the best use of editorial resources.

**Rank 13** and **Rank 15** are also worth treating cautiously. Both have been unchanged for 301 days, which makes them clearly stale according to the rule, but their impression counts are only 954 and 821 respectively. Their age provides a reason to investigate them, but it does not establish that updating them will produce meaningful improvement.

There is also an important example in the opposite direction. **Rank 18** has more than 517,000 impressions but receives `NO_ACTION` because it was updated only 104 days ago. Ranks 19 and 20 have more than 509,000 impressions each but were updated only 22 and 20 days ago. These are not necessarily bad recommendations by the baseline, but they demonstrate a limitation: the rule can miss highly visible pages that may need attention for reasons other than staleness.

Overall, the weakest positive recommendations are the pages that barely pass the visibility threshold. The baseline is useful for creating a transparent review queue, but it does not consider content quality, search intent, seasonality, or other reasons why a page may or may not benefit from a refresh.

### Leakage check

The baseline uses only `days_since_last_update` and `impressions_90d` to calculate the score. I did not use future-window measurements or label-derived fields. In particular, I did not use `is_declining_label`, `trend_direction`, or `trend_pct` as inputs to the rule.

I also did not use existing product decisions, flags, priority scores, or action labels to generate the baseline. The pseudonymized identifiers are used only to identify rows in the output and are not used to calculate the score.

Therefore, the baseline is an independently defined decision-support rule rather than a reconstruction of an existing product decision.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.